# Extension: China Economic Policy Uncertainty (EPU)
**Paper**: Baker et al. (2016) - Measuring Economic Policy Uncertainty
**Data**: China EPU Index (local file) + China macro data (local CSV files)

**Key Question**: How does policy uncertainty affect China's investment, consumption, and employment?

**Data Status**: All data loaded from local files in the data/ directory.

## 1. Setup and Data Download

In [ ]:
# Download data files if not already present
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/JasmineHao/JasmineHao.github.io/main/econ6083/final-project/notebooks/data/"
DATA_FILES = ['china_epu.csv', 'china_cpi.csv', 'china_pmi.csv', 'china_lpr.csv', 'us_epu_daily.csv']

os.makedirs('data', exist_ok=True)
for fname in DATA_FILES:
    if not os.path.exists(f'data/{fname}'):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, f'data/{fname}')
        print(f"  Saved to data/{fname}")
    else:
        print(f"Found local: data/{fname}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Load China EPU data from local file
# Monthly China EPU index (Baker et al. 2016)
china_epu = pd.read_csv('data/china_epu.csv')
china_epu['date'] = pd.to_datetime(china_epu[['year', 'month']].assign(day=1))
china_epu = china_epu.rename(columns={'China_Policy_Index': 'epu'})

print(f"China EPU loaded: {china_epu.shape}")
print(f"Date range: {china_epu['date'].min()} to {china_epu['date'].max()}")
print(china_epu.head())
print(china_epu.tail())

## 2. Download China Macroeconomic Data (local CSV files)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Load China macro data from local files
# CPI (Consumer Price Index year-over-year)
cpi = pd.read_csv('data/china_cpi.csv')
cpi['date'] = pd.to_datetime(cpi['date'])

# PMI (Purchasing Managers' Index)
pmi = pd.read_csv('data/china_pmi.csv')
pmi['date'] = pd.to_datetime(pmi['date'])

# LPR (Loan Prime Rate, 1-year)
lpr = pd.read_csv('data/china_lpr.csv')
lpr['date'] = pd.to_datetime(lpr['date'])

print(f"CPI: {cpi.shape}, PMI: {pmi.shape}, LPR: {lpr.shape}")
print("\nCPI head:")
print(cpi.head(3))
print("\nPMI head:")
print(pmi.head(3))
print("\nLPR head:")
print(lpr.head(3))

## 3. Merge Data and Explore

In [ ]:
# Convert all to year_month period for merging
china_epu['year_month'] = china_epu['date'].dt.to_period('M')
cpi['year_month'] = cpi['date'].dt.to_period('M')
pmi['year_month'] = pmi['date'].dt.to_period('M')
lpr_monthly['year_month'] = lpr_monthly['date'].dt.to_period('M')

# Merge
df = china_epu[['year_month', 'epu']].merge(
    cpi[['year_month', 'cpi_yoy']], on='year_month', how='inner'
)
df = df.merge(pmi[['year_month', 'pmi']], on='year_month', how='inner')
df = df.merge(lpr_monthly[['year_month', 'lpr_1y']], on='year_month', how='inner')

print(f"Merged dataset shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")
print("\nCorrelations with China EPU:")
print(df[['epu', 'cpi_yoy', 'pmi', 'lpr_1y']].corr()['epu'].round(3))

df.head()

## 4. Regression: EPU Effects on China Macro Outcomes

In [ ]:
# Define outcomes
outcomes = ['pmi', 'lpr_1y']
results = {}

for outcome in outcomes:
    corr = df['epu'].corr(df[outcome])
    
    # OLS
    df_reg = df[['epu', outcome]].dropna()
    X = sm.add_constant(df_reg['epu'])
    model = sm.OLS(df_reg[outcome], X).fit()
    
    # Lagged (3 months)
    df['epu_lag3'] = df['epu'].shift(3)
    df_lag = df[['epu_lag3', outcome]].dropna()
    X_lag = sm.add_constant(df_lag['epu_lag3'])
    model_lag = sm.OLS(df_lag[outcome], X_lag).fit()
    
    results[outcome] = {
        'corr': corr,
        'coef': model.params[1],
        'pvalue': model.pvalues[1],
        'r2': model.rsquared,
        'coef_lag3': model_lag.params[1],
        'r2_lag3': model_lag.rsquared
    }

results_df = pd.DataFrame(results).T
print("China EPU Effects:")
print("="*60)
print(results_df.round(4))

## 5. Visualization

In [ ]:
df['date'] = df['year_month'].dt.to_timestamp()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# China EPU time series
ax1 = axes[0, 0]
ax1.plot(df['date'], df['epu'], color='red', linewidth=1)
ax1.set_ylabel('China EPU Index')
ax1.set_title('China Economic Policy Uncertainty Index')
ax1.grid(alpha=0.3)

# EPU vs PMI
ax2 = axes[0, 1]
ax2.scatter(df['epu'], df['pmi'], alpha=0.5, color='blue')
df_plot = df[['epu', 'pmi']].dropna()
z = np.polyfit(df_plot['epu'], df_plot['pmi'], 1)
p = np.poly1d(z)
ax2.plot(df_plot['epu'], p(df_plot['epu']), "r--", alpha=0.8)
ax2.set_xlabel('China EPU')
ax2.set_ylabel('PMI')
ax2.set_title('China EPU vs Manufacturing PMI')
ax2.grid(alpha=0.3)

# EPU vs LPR
ax3 = axes[1, 0]
ax3.scatter(df['epu'], df['lpr_1y'], alpha=0.5, color='green')
df_plot2 = df[['epu', 'lpr_1y']].dropna()
z = np.polyfit(df_plot2['epu'], df_plot2['lpr_1y'], 1)
p = np.poly1d(z)
ax3.plot(df_plot2['epu'], p(df_plot2['epu']), "r--", alpha=0.8)
ax3.set_xlabel('China EPU')
ax3.set_ylabel('LPR 1-Year (%)')
ax3.set_title('China EPU vs Loan Prime Rate')
ax3.grid(alpha=0.3)

# Rolling correlation
ax4 = axes[1, 1]
rolling_corr = df['epu'].rolling(window=24).corr(df['pmi'])
ax4.plot(df['date'], rolling_corr, color='purple')
ax4.set_xlabel('Year')
ax4.set_ylabel('Rolling Correlation (24-month)')
ax4.set_title('EPU-PMI Correlation Over Time')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Compare US vs China EPU (Optional)

In [ ]:
# Download US monthly EPU for comparison
us_epu = pd.read_csv('data/us_epu_daily.csv')
us_epu['date'] = pd.to_datetime(us_epu[['Year', 'Month']].assign(day=1))
us_epu = us_epu.rename(columns={'News_Based_Policy_Uncert_Index': 'epu_us'})
us_epu['year_month'] = us_epu['date'].dt.to_period('M')

# Merge US and China EPU
compare = china_epu[['year_month', 'epu']].merge(
    us_epu[['year_month', 'epu_us']], on='year_month', how='inner'
)
compare['date'] = compare['year_month'].dt.to_timestamp()

print(f"Overlap period: {compare['year_month'].min()} to {compare['year_month'].max()}")
print(f"Correlation (US vs China EPU): {compare['epu'].corr(compare['epu_us']):.3f}")

# Plot comparison
plt.figure(figsize=(12, 5))
plt.plot(compare['date'], compare['epu'], label='China EPU', color='red')
plt.plot(compare['date'], compare['epu_us'], label='US EPU', color='blue', alpha=0.7)
plt.ylabel('EPU Index')
plt.title('US vs China Economic Policy Uncertainty')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Interpreting Your Results

| Output | What it means | What to look for |
|---|---|---|
| **China EPU trend** | Monthly policy uncertainty in China | Compare pre/post 2008 crisis, pre/post trade war |
| **Correlation with CPI/PMI** | Does EPU predict inflation/business activity? | Negative with PMI = uncertainty dampens manufacturing |
| **US vs China EPU** | Synchronization or divergence | Do the two economies co-move in uncertainty? |
| **LPR response** | Does EPU affect interest-rate setting? | Positive = PBoC eases when uncertainty rises |

**Key question**: Is China's EPU driven by domestic policy (e.g., regulatory crackdowns) or global shocks (e.g., US trade policy)? The comparison with US EPU helps disentangle this.

## Summary

This notebook demonstrates a **fully runnable China EPU extension** using only publicly available data:

1. **China EPU**: Directly downloaded from local file
2. **China Macro Data**: CPI, PMI, LPR via local CSV files (free)
3. **Analysis**: OLS + lagged regressions identical to the US version
4. **Comparison**: US vs China EPU correlation analysis

**Key Finding**: Students should summarize their main empirical finding here.

**Further Extensions**:
- Add more macro variables (fixed asset investment, retail sales, M2)
- Use local projections (Jorda 2005) for dynamic effects
- Separate analysis for monetary policy vs fiscal policy uncertainty
- Compare EPU effects across industry sectors (using sector-level PMI)